In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/csafrit2/maternal-health-risk-data/Maternal Health Risk Data Set.csv


In [5]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Load Data
maternal_df = pd.read_csv('/kaggle/input/datasets/csafrit2/maternal-health-risk-data/Maternal Health Risk Data Set.csv')

# Calculate quick summary metrics for our KPI cards
total_patients = len(maternal_df)
avg_age = round(maternal_df['Age'].mean(), 1)
high_risk_pct = round((len(maternal_df[maternal_df['RiskLevel'] == 'high risk']) / total_patients) * 100, 1)

# 2. Create a Grid Layout (2 Rows, 2 Columns for charts + KPI space)
fig = make_subplots(
    rows=3, cols=2,
    row_heights=[0.2, 0.4, 0.4],  # Row 1 is for KPIs, Rows 2 & 3 for charts
    specs=[[{"type": "indicator"}, {"type": "indicator"}],  # Top KPI row
           [{"type": "xy"}, {"type": "domain"}],             # Middle charts
           [{"type": "xy"}, {"type": "xy"}]],                # Bottom charts
    subplot_titles=("", "", 
                    "Age vs. Systolic Blood Pressure", "Risk Level Distribution", 
                    "Blood Sugar Levels by Risk", "Heart Rate Trends")
)

# --- ROW 1: KPI CARDS ---
fig.add_trace(go.Indicator(
    mode="number", value=total_patients,
    title={"text": "Total Patients Analyzed"},
    domain={'x': [0, 0.45], 'y': [0.8, 1]}
), row=1, col=1)

fig.add_trace(go.Indicator(
    mode="number", value=high_risk_pct, number={'suffix': "%"},
    title={"text": "High Risk Prevalence"},
    domain={'x': [0.55, 1], 'y': [0.8, 1]}
), row=1, col=2)

# --- ROW 2: CHARTS ---
# Chart 1: Scatter (Age vs Blood Pressure)
for risk, group in maternal_df.groupby('RiskLevel'):
    fig.add_trace(go.Scatter(
        x=group['Age'], y=group['SystolicBP'], mode='markers', name=risk
    ), row=2, col=1)

# Chart 2: Donut Chart (Risk Level breakdown)
risk_counts = maternal_df['RiskLevel'].value_counts()
fig.add_trace(go.Pie(
    labels=risk_counts.index, values=risk_counts.values, hole=0.4
), row=2, col=2)

# --- ROW 3: CHARTS ---
# Chart 3: Bar Chart (Blood Sugar by Risk Level)
avg_bs = maternal_df.groupby('RiskLevel')['BS'].mean().reset_index()
fig.add_trace(go.Bar(
    x=avg_bs['RiskLevel'], y=avg_bs['BS'], marker_color='#9e1b1b'
), row=3, col=1)

# Chart 4: Histogram (Age distribution)
fig.add_trace(go.Histogram(
    x=maternal_df['Age'], nbinsx=20, marker_color='#4a4a4a'
), row=3, col=2)

# 3. Design & Styling (Matching the clean, professional look)
fig.update_layout(
    title_text="🧠 Maternal Health Executive Risk Analysis Dashboard",
    template="plotly_white",
    height=900,  # Forces everything into a readable frame height
    showlegend=True,
    margin=dict(t=80, b=40, l=40, r=40)
)

fig.show()

In [6]:
# Save the whole dashboard grid as a single offline web app file
fig.write_html("Maternal_Health_Executive_Dashboard.html")